# Naija-Speech — TTS Data Audit (Phase 0 of the TTS track)

Answers the three questions the TTS fine-tunes depend on — **without re-downloading
the 70 GB corpus**:

1. **Speaker backfill** — can AfriSpeech-200 rows recover speaker IDs by joining the
   source manifest on transcript text? (Decides Orpheus's speaker strategy.)
   Uses DuckDB *columnar* reads: only the text/accent/duration columns come over the
   network (~MBs).
2. **Sample rates** — what did "store native" actually store? (StyleTTS 2 trains at
   24 kHz; clips recorded below that can't be upsampled into quality.)
   Header-only reads on a streamed sample.
3. **Quality heuristics** — clipping / silence / dynamic-range proxies per
   source+domain, to shape the TTS-grade selection. *Heuristic* — the real UTMOSv2
   scoring runs on GPU at selection time.

Thin notebook: all logic lives in `scripts/07_tts_data_audit.py`; cells call its
functions so results display inline. Runs fine on CPU; ~5–15 min total.

In [ ]:
import os, sys

if not os.path.exists("scripts") and os.path.exists("../scripts"):
    os.chdir("..")
sys.path.insert(0, "src")
print("repo root:", os.getcwd())

## 0 — Install dependencies (same cell as notebook 03; idempotent)

In [ ]:
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

need_cuda_torch = importlib.util.find_spec("torch") is None
if not need_cuda_torch:
    import torch
    need_cuda_torch = not torch.cuda.is_available()
if need_cuda_torch:
    print("installing CUDA torch (~2.5 GB, one time) ...")
    pip("torch", "--index-url", "https://download.pytorch.org/whl/cu124")

pip("-r", "requirements.txt")   # includes duckdb for the audit

print("kernel env:", sys.executable)   # MUST be the project .venv

## 1 — Load the audit module + sanity self-test (no network)

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("tts_audit", "scripts/07_tts_data_audit.py")
audit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)

from config import load_dotenv, load_yaml
load_dotenv()
cfg = load_yaml("configs/data_afrispeech_ng.yaml")

audit.self_test()   # verifies the join tiers + quality heuristics on synthetic data

## 2 — Part A: speaker backfill (metadata-only, ~seconds of transfer)

Reads only text/accent/duration columns from the remote parquet shards, then joins
against the AfriSpeech-200 transcript manifest in three tiers (unique text →
text+accent+duration → ambiguous/unmatched).

**How to read the result:** the recoverable % decides Orpheus's speaker strategy —
high (say ≥80 %) → Plan A, backfilled speaker IDs; low → Plan B, Hypa-style named
voices.

In [ ]:
speaker_counts, speaker_total, annotated = audit.audit_speakers(cfg)
annotated.drop(columns=["key"]).head(10)

## 3 — Parts B + C: sample rates & quality heuristics (streamed sample)

Adjust the knobs if you want a bigger/smaller sample. ~1–2 GB streams for the
default 1,000 headers; 300 of those decode fully for the quality proxies.

In [ ]:
SAMPLE_N = 1000   # clips streamed for header stats (sample rate / channels)
DECODE_N = 300    # of those, fully decoded for quality heuristics

audio_df = audit.audit_audio(cfg, SAMPLE_N, DECODE_N)
audio_df.head()

In [ ]:
# Inline views: sample-rate histogram + quality by source/domain.
import matplotlib.pyplot as plt

ax = audio_df.groupby("samplerate").size().plot(kind="bar", figsize=(7, 3.5),
                                                title="Native sample rates (streamed sample)")
ax.set_ylabel("clips"); plt.tight_layout(); plt.show()

q = audio_df.dropna(subset=["dyn_range_db"]) if "dyn_range_db" in audio_df else None
if q is not None and len(q):
    display(q.groupby(["source", "domain"])[
        ["clipping_pct", "silence_pct", "dyn_range_db"]].mean().round(1))

## 4 — Write the report

Produces `outputs/tts_audit/tts_audit_report.md` (+ `speaker_backfill.csv`,
`audio_sample_stats.csv`). **Paste the report back into the chat** — it decides:
1. Orpheus speaker strategy (Plan A backfill vs Plan B named voices),
2. StyleTTS 2 data-selection criteria (which source/domain slices are TTS-grade),
3. whether sample rates force any change to the 24 kHz training target.

In [ ]:
audit.write_report(cfg, speaker_counts, speaker_total, audio_df)
print(open("outputs/tts_audit/tts_audit_report.md", encoding="utf-8").read())